In [1]:
# import required Libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

print("TF:", tf.__version__)
print("PD:", pd.__version__)

TF: 2.19.0
PD: 2.2.2


In [2]:
# Step 1: Loading telco.csv (attached dataset)

csv_path = "telco.csv"   # making sure this matches the uploaded filename

df = pd.read_csv(csv_path)

df.shape        # (rows, columns)

(7043, 50)

In [3]:
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [4]:
print("\nInfo: (null values, data types)")
df.info()


Info: (null values, data types)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   object 
 4   Senior Citizen                     7043 non-null   object 
 5   Married                            7043 non-null   object 
 6   Dependents                         7043 non-null   object 
 7   Number of Dependents               7043 non-null   int64  
 8   Country                            7043 non-null   object 
 9   State                              7043 non-null   object 
 10  City                               7043 non-null   object 
 11  Zip Code               

In [5]:
print("\nChurn Label value counts:")
df["Churn Label"].value_counts()


Churn Label value counts:


,count
Churn Label,
No,5174
Yes,1869


In [6]:
# Step 2: Creating numeric target column (churn from churn label) and drop ID + Churn label

target_col = "Churn"  # name for our numeric target column

# 1) Create numeric 'Churn' from text 'Churn Label' ("Yes"/"No" -> 1/0)
# we need to convert strings to integers for the model.
df[target_col] = df["Churn Label"].map({"Yes": 1, "No": 0}).astype("int32")

# 2) Drop 'Customer ID' (identifier, not a useful feature) AND 'Churn Label' (text version of target)
# we now explicitly drop 'Churn Label' so that it is NOT used as a feature.
df = df.drop(columns=["Customer ID", "Churn Label"])

In [7]:
# Step 3: Identify numeric and categorical columns

# Numeric columns: int64/float64, EXCLUDING the target
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]

# Categorical columns: object (string) columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("\nCategorical columns:", categorical_cols)

Numeric columns: ['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Satisfaction Score', 'Churn Score', 'CLTV']

Categorical columns: ['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Country', 'State', 'City', 'Quarter', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Customer Status', 'Churn Category', 'Churn Reason']


In [8]:
# Step 4: Handle missing values BEFORE encoding
# ----------------------------------------------------------------
# Numeric columns:
#   - fill NaN with the column median (simple, robust choice).
# Categorical columns:
#   - fill NaN with a special category "Unknown" so get_dummies can handle it.

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [9]:
# Step 5: One-hot encode categorical features
#   - get_dummies turns each category into separate 0/1 columns.

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded shape:", df_encoded.shape)

Encoded shape: (7043, 1182)


In [10]:
# step 6: taking numerical column, encoded categorical column and encoded target column separately

# Numeric features: from original df, but after NaN handling
X_num = df[numeric_cols].values.astype("float32")  # only numeric columns

# Categorical (one-hot) features:
#  - df_encoded has all columns including numeric, one-hot, and 'Churn'
#  - we remove numeric_cols and target_col; the rest are one-hot cat columns. [web:114]
one_hot_cols = [c for c in df_encoded.columns if c not in numeric_cols + [target_col]]
X_cat = df_encoded[one_hot_cols].values.astype("float32")

y = df_encoded[target_col].values.astype("int32")

print("X_num shape:", X_num.shape)        # (N, num_numeric_features)
print("X_cat shape:", X_cat.shape)        # (N, num_one_hot_cat_features)
print("y shape:", y.shape)

X_num shape: (7043, 19)
X_cat shape: (7043, 1162)
y shape: (7043,)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 7: Split into train/test for BOTH numeric and categorical branches together

X_num_train, X_num_test, X_cat_train, X_cat_test, y_train, y_test = train_test_split(
    X_num,
    X_cat,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_num_train shape:", X_num_train.shape)
print("X_cat_train shape:", X_cat_train.shape)
print("y_train shape:", y_train.shape)
print("X_num_test shape:", X_num_test.shape)
print("X_cat_test shape:", X_cat_test.shape)
print("y_test shape:", y_test.shape)

X_num_train shape: (5634, 19)
X_cat_train shape: (5634, 1162)
y_train shape: (5634,)
X_num_test shape: (1409, 19)
X_cat_test shape: (1409, 1162)
y_test shape: (1409,)


In [12]:
# Step 8: Scale numeric features only

num_scaler = StandardScaler()
X_num_train_scaled = num_scaler.fit_transform(X_num_train)
X_num_test_scaled  = num_scaler.transform(X_num_test)

print("Scaled numeric train mean:", X_num_train_scaled.mean(axis=0)[:5])
print("Scaled numeric train std :", X_num_train_scaled.std(axis=0)[:5])

Scaled numeric train mean: [-2.7347889e-08  2.9969479e-07 -3.4383225e-09  1.5255573e-08
  6.3899903e-09]
Scaled numeric train std : [0.999998  0.9999897 1.        0.9999992 1.0000002]


**Running the Functional API model with np.array form dataset**

In [13]:
from tensorflow.keras import regularizers

# Step 9: Build multi-input Functional model (numeric + categorical branches)

# Inputs
num_input = tf.keras.Input(shape=(X_num_train_scaled.shape[1],), name="numeric_input")
cat_input = tf.keras.Input(shape=(X_cat_train.shape[1],), name="categorical_input")

# Numeric branch
#  - Process numeric features with a small Dense sub-network.
x_num = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
    )(num_input)

x_num = layers.Dropout(0.3)(x_num)

# Categorical branch
#  - Process one-hot categorical features separately.
x_cat = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
    )(cat_input)

x_cat = layers.Dropout(0.3)(x_cat)

# Concatenate branches
#  - Combine learned numeric and categorical representations.
x = layers.Concatenate()([x_num, x_cat])

# Shared layers after concatenation
x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
    )(x)

x = layers.Dropout(0.3)(x)

# Output layer
output = layers.Dense(1, activation="sigmoid")(x)

# Build the model with MULTIPLE inputs and ONE output
multi_input_model = tf.keras.Model(
    inputs=[num_input, cat_input],
    outputs=output,
    name="telco_multi_input_model"
    )

multi_input_model.summary()

Model: "telco_multi_input_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ numeric_input       │ (None, 19)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ categorical_input   │ (None, 1162)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        640 │ numeric_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     74,432 │ categorical_inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 96)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      3,104 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 78,209 (305.50 KB)

 Trainable params: 78,209 (305.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Step 10: Compile the model
multi_input_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
)

early_stop_multi = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint_multi = ModelCheckpoint(
    "best_telco_multi_input.keras",
    monitor="val_loss",
    save_best_only=True
)

history_multi = multi_input_model.fit(
    [X_num_train_scaled, X_cat_train],  # list of inputs: [numeric, categorical]
    y_train,
    epochs=40,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop_multi, checkpoint_multi],
    verbose=2
)

Epoch 1/40
71/71 - 5s - 74ms/step - accuracy: 0.8371 - loss: 0.4685 - val_accuracy: 0.9965 - val_loss: 0.1377
Epoch 2/40
71/71 - 1s - 11ms/step - accuracy: 0.9978 - loss: 0.1093 - val_accuracy: 1.0000 - val_loss: 0.0770
Epoch 3/40
71/71 - 0s - 7ms/step - accuracy: 1.0000 - loss: 0.0758 - val_accuracy: 1.0000 - val_loss: 0.0648
Epoch 4/40
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0640 - val_accuracy: 1.0000 - val_loss: 0.0563
Epoch 5/40
71/71 - 0s - 6ms/step - accuracy: 0.9998 - loss: 0.0554 - val_accuracy: 1.0000 - val_loss: 0.0492
Epoch 6/40
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0493 - val_accuracy: 1.0000 - val_loss: 0.0434
Epoch 7/40
71/71 - 0s - 7ms/step - accuracy: 1.0000 - loss: 0.0431 - val_accuracy: 1.0000 - val_loss: 0.0382
Epoch 8/40
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0382 - val_accuracy: 1.0000 - val_loss: 0.0337
Epoch 9/40
71/71 - 0s - 7ms/step - accuracy: 1.0000 - loss: 0.0343 - val_accuracy: 1.0000 - val_loss: 0.0301
Epoch 10/40
71/71

In [15]:
# Step 11: Evaluate the multi-input model on the test set

test_loss_multi, test_acc_multi = multi_input_model.evaluate(
    [X_num_test_scaled, X_cat_test],
    y_test,
    verbose=2
)

print("Multi-input model - Test loss:", test_loss_multi)
print("Multi-input model - Test accuracy:", test_acc_multi)

45/45 - 0s - 3ms/step - accuracy: 1.0000 - loss: 0.0060
Multi-input model - Test loss: 0.005955204833298922
Multi-input model - Test accuracy: 1.0


**Running The Functional API model with tf.data.dataset form dataset**

In [16]:
# Step 9: Build tf.data Datasets for multi-input model

batch_size = 64

# 1) split X_num_train_scaled and X_cat_train into Train/val
from sklearn.model_selection import train_test_split

X_num_train_sub, X_num_val, X_cat_train_sub, X_cat_val, y_train_sub, y_val = train_test_split(
    X_num_train_scaled,
    X_cat_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("X_num_train_sub:", X_num_train_sub.shape)
print("X_cat_train_sub:", X_cat_train_sub.shape)
print("y_train_sub:", y_train_sub.shape)
print("X_num_val:", X_num_val.shape)
print("X_cat_val:", X_cat_val.shape)
print("y_val:", y_val.shape)

X_num_train_sub: (4507, 19)
X_cat_train_sub: (4507, 1162)
y_train_sub: (4507,)
X_num_val: (1127, 19)
X_cat_val: (1127, 1162)
y_val: (1127,)


In [17]:
# step 10: creating dataset pipeline using tf.data.dataframe
# Training dataset:
# Each element is: ( [numeric_features_row, categorical_features_row], label )
train_multi_ds = tf.data.Dataset.from_tensor_slices(
    ((X_num_train_sub, X_cat_train_sub), y_train_sub))

train_multi_ds = (
    train_multi_ds
    .shuffle(buffer_size=len(X_num_train_sub))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
    )

# Validation dataset:
val_multi_ds = tf.data.Dataset.from_tensor_slices(
    ((X_num_val, X_cat_val), y_val))

val_multi_ds = val_multi_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Test dataset:
test_multi_ds = tf.data.Dataset.from_tensor_slices(
    ((X_num_test_scaled, X_cat_test), y_test))

test_multi_ds = test_multi_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

**Creating model with the Sequential model and the functional API combined**

In [20]:
# step 11: creating the model
# Numeric branch as Sequential
numeric_branch = tf.keras.Sequential([
    layers.Input(shape=(X_num_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
])

# Categorical branch as Sequential
cat_branch = tf.keras.Sequential([
    layers.Input(shape=(X_cat_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
])

# Wrap them with Functional API
num_input = tf.keras.Input(shape=(X_num_train_scaled.shape[1],))
cat_input = tf.keras.Input(shape=(X_cat_train.shape[1],))

x_num = numeric_branch(num_input)
x_cat = cat_branch(cat_input)

x = layers.Concatenate()([x_num, x_cat])
x = layers.Dense(32, activation="relu")(x)
output = layers.Dense(1, activation="sigmoid")(x)

multi_input_model = tf.keras.Model(inputs=[num_input, cat_input], outputs=output)

In [18]:
# step 12: compile the model

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

multi_input_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
    )

early_stop_multi = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
    )

checkpoint_multi = ModelCheckpoint(
    "best_telco_multi_input_tfdata.keras",
    monitor="val_loss",
    save_best_only=True
    )

# Train with tf.data Datasets instead of NumPy
history_multi_ds = multi_input_model.fit(
    train_multi_ds,
    epochs=40,
    validation_data=val_multi_ds,
    callbacks=[early_stop_multi, checkpoint_multi],
    verbose=2
    )

Epoch 1/40
71/71 - 4s - 61ms/step - accuracy: 1.0000 - loss: 0.0068 - val_accuracy: 1.0000 - val_loss: 0.0057
Epoch 2/40
71/71 - 1s - 12ms/step - accuracy: 1.0000 - loss: 0.0063 - val_accuracy: 1.0000 - val_loss: 0.0053
Epoch 3/40
71/71 - 1s - 11ms/step - accuracy: 1.0000 - loss: 0.0060 - val_accuracy: 1.0000 - val_loss: 0.0050
Epoch 4/40
71/71 - 1s - 10ms/step - accuracy: 1.0000 - loss: 0.0056 - val_accuracy: 1.0000 - val_loss: 0.0049
Epoch 5/40
71/71 - 1s - 11ms/step - accuracy: 1.0000 - loss: 0.0057 - val_accuracy: 1.0000 - val_loss: 0.0048
Epoch 6/40
71/71 - 1s - 17ms/step - accuracy: 1.0000 - loss: 0.0053 - val_accuracy: 1.0000 - val_loss: 0.0044
Epoch 7/40
71/71 - 0s - 5ms/step - accuracy: 1.0000 - loss: 0.0053 - val_accuracy: 1.0000 - val_loss: 0.0048
Epoch 8/40
71/71 - 1s - 9ms/step - accuracy: 1.0000 - loss: 0.0050 - val_accuracy: 1.0000 - val_loss: 0.0042
Epoch 9/40
71/71 - 1s - 9ms/step - accuracy: 1.0000 - loss: 0.0049 - val_accuracy: 1.0000 - val_loss: 0.0044
Epoch 10/40
7

In [19]:
# step 13: Evaluate on test dataset
test_loss_multi_ds, test_acc_multi_ds = multi_input_model.evaluate(
    test_multi_ds,
    verbose=2
    )

print("Multi-input model (tf.data) - Test loss:", test_loss_multi_ds)
print("Multi-input model (tf.data) - Test accuracy:", test_acc_multi_ds)

23/23 - 0s - 12ms/step - accuracy: 1.0000 - loss: 0.0025
Multi-input model (tf.data) - Test loss: 0.0024891686625778675
Multi-input model (tf.data) - Test accuracy: 1.0
